In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-beta")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

model-00003-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

c:\Users\chand.CHANDAN\anaconda3\envs\aiche-llm\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chand.CHANDAN\.cache\huggingface\hub\models--HuggingFaceH4--zephyr-7b-beta. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model-00004-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

In [ ]:
import json
import re
from typing import List, Dict, Any
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True  # REQUIRED when some layers spill to CPU
)

# =========================
# Model (Mistral Instruct)
# =========================
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    load_in_8bit=True, 
    llm_int8_enable_fp32_cpu_offload=True,
    trust_remote_code=True,
)
gen = pipeline("text-generation", model=model, tokenizer=tokenizer)

# =========================
# Prompt (5 keywords only)
# =========================
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5 to 10 unique entries without explanation.
"""

# =========================
# Few-shot (adapted)
# =========================
FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

# =========================
# User Prompt Template
# =========================
User_Prompt_template = """
{few_shot}
Extract exactly 5 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

GENERIC = {
    "catalysis", "analysis", "study", "experiment", "experimental",
    "characterization", "materials", "material", "chemical engineering",
    "technique", "techniques", "process", "processes", "method", "methods",
    "optimization", "performance", "investigation", "properties", "system",
    "approach", "results", "paper", "model", "models"
}
KEEP_IF_CONTAINS = ["heterogeneous catalysis", "homogeneous catalysis"]

STOPWORDS = set("""
a an the and or for with of on into to from via in at by using use over under between among
this that these those our their your its is are was were be being been have has had will would
we they it as than also may can could should other more less based derived new novel toward towards
""".split())

def _apply_chat(system: str, user: str) -> str:
    msgs = [
        {"role": "system", "content": system.strip()},
        {"role": "user", "content": user.strip()},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _extract_json(text: str) -> Dict[str, Any]:
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return {}
    block = m.group(0).strip().strip("`")
    try:
        return json.loads(block)
    except Exception:
        block = re.sub(r"'", '"', block)
        block = re.sub(r",\s*]", "]", block)
        block = re.sub(r",\s*}", "}", block)
        try:
            return json.loads(block)
        except Exception:
            return {}

def _clean_list(xs) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if not k:
            continue
        lo = k.lower()
        if lo not in seen:
            seen.add(lo)
            out.append(k)
    return out

def _filter_generic(term: str) -> bool:
    lo = term.lower()
    if any(s in lo for s in KEEP_IF_CONTAINS):
        return True
    return lo not in GENERIC and len(lo) >= 3

def _authors_block(item: Dict[str, Any]) -> str:
    if not isinstance(item, dict):
        return ""
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm = (a.get("name") or "").strip()
            af = (a.get("affiliation") or "").strip()
            if nm or af:
                parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)[:800]

# =========================
# Core extraction
# =========================
def extract_keywords_and_countries(title: str, abstract_text: str, authors_txt: str,
                                   max_new_tokens: int = 128) -> Dict[str, List[str]]:
    user = User_Prompt_template.format(
        few_shot=FEW_SHOT.strip(),
        title_or_topic=(title or "").strip(),
        authors_block=(authors_txt or "").strip(),
        abstract_text=(abstract_text or "").strip()
    )
    prompt = _apply_chat(System_Prompt, user)

    out = gen(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )[0]["generated_text"]

    data = _extract_json(out) or {}
    keywords = _clean_list(data.get("keywords", []))
    keywords = [k for k in keywords if k and _filter_generic(k)]

    countries = _clean_list(data.get("countries", []))
    return {"keywords": keywords, "countries": countries}

def process_file(path: str) -> List[Dict[str, List[str]]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Input JSON must be a list of objects.")

    results = []
    for item in data:
        if isinstance(item, dict):
            title = item.get("title") or item.get("topic") or ""
            abstract_text = item.get("abstract") or ""
            authors_txt = _authors_block(item)
        else:
            title, abstract_text, authors_txt = "", str(item), ""
        results.append(extract_keywords_and_countries(title, abstract_text, authors_txt))
    return results

def results_keywords_only(results: List[Dict[str, List[str]]]) -> List[List[str]]:
    return [r["keywords"] for r in results]

# =========================
# MAIN
# =========================
OUTPUT_DIR = "openrouter/extracted"
os.makedirs(OUTPUT_DIR, exist_ok=True)

filename = "aiche_sample.json"
save_full_results_to = os.path.join(OUTPUT_DIR, "full_results.json")
save_keywords_only_to = os.path.join(OUTPUT_DIR, "keywords_only.json")

if __name__ == "__main__":
    all_results = process_file(filename)

    print("FULL RESULTS:")
    print(json.dumps(all_results, ensure_ascii=False, indent=2))

    if save_full_results_to:
        with open(save_full_results_to, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

    kw_matrix = results_keywords_only(all_results)
    print("\nKEYWORDS ONLY (array of arrays):")
    print(json.dumps(kw_matrix, ensure_ascii=False, indent=2))

    if save_keywords_only_to:
        with open(save_keywords_only_to, "w", encoding="utf-8") as f:
            json.dump(kw_matrix, f, ensure_ascii=False, indent=2)


tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\chand.CHANDAN\anaconda3\envs\aiche-llm\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chand.CHANDAN\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

In [ ]:
import json
import re
from typing import List, Dict, Any
import ollama   # ✅ Ollama client

# =========================
# Model config (Llama 3 via Ollama)
# =========================
OLLAMA_MODEL = "llama3:instruct"  # or "llama3:instruct-q4_K_M" for lower VRAM

System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5–10 unique entries without explanation.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

GENERIC = {
    "catalysis", "analysis", "study", "experiment", "experimental",
    "characterization", "materials", "material", "chemical engineering",
    "technique", "techniques", "process", "processes", "method", "methods",
    "optimization", "performance", "investigation", "properties", "system",
    "approach", "results", "paper", "model", "models"
}
KEEP_IF_CONTAINS = ["heterogeneous catalysis", "homogeneous catalysis"]

STOPWORDS = set("""
a an the and or for with of on into to from via in at by using use over under between among
this that these those our their your its is are was were be being been have has had will would
we they it as than also may can could should other more less based derived new novel toward towards
""".split())

# ===== Helpers =====
def _apply_chat(system: str, user: str) -> list:
    return [
        {"role": "system", "content": system.strip()},
        {"role": "user", "content": user.strip()},
    ]

def _extract_json(text: str) -> Dict[str, Any]:
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return {}
    block = m.group(0).strip().strip("`")
    try:
        return json.loads(block)
    except Exception:
        block = re.sub(r"'", '"', block)
        block = re.sub(r",\s*]", "]", block)
        block = re.sub(r",\s*}", "}", block)
        try:
            return json.loads(block)
        except Exception:
            return {}

def _clean_list(xs) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if not k:
            continue
        lo = k.lower()
        if lo not in seen:
            seen.add(lo)
            out.append(k)
    return out

def _filter_generic(term: str) -> bool:
    lo = term.lower()
    if any(s in lo for s in KEEP_IF_CONTAINS):
        return True
    return lo not in GENERIC and len(lo) >= 3

def _authors_block(item: Dict[str, Any]) -> str:
    if not isinstance(item, dict):
        return ""
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm = (a.get("name") or "").strip()
            af = (a.get("affiliation") or "").strip()
            if nm or af:
                parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)[:800]

# ===== Ollama chat call =====
def _ollama_chat_once(messages: list) -> str:
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=messages,
        options={"temperature": 0},
        keep_alive="10m"
    )
    return resp["message"]["content"]

# ===== Core extraction =====
def extract_keywords_and_countries(title: str, abstract_text: str, authors_txt: str) -> Dict[str, List[str]]:
    user = User_Prompt_template.format(
        few_shot=FEW_SHOT.strip(),
        title_or_topic=(title or "").strip(),
        authors_block=(authors_txt or "").strip(),
        abstract_text=(abstract_text or "").strip()
    )
    messages = _apply_chat(System_Prompt, user)
    out = _ollama_chat_once(messages)

    data = _extract_json(out) or {}
    keywords = _clean_list(data.get("keywords", []))
    keywords = [k for k in keywords if k and _filter_generic(k)]
    countries = _clean_list(data.get("countries", []))
    return {"keywords": keywords, "countries": countries}

# ===== File handling =====
def process_file(path: str) -> List[Dict[str, List[str]]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Input JSON must be a list of objects.")

    results = []
    for item in data:
        if isinstance(item, dict):
            title = item.get("title") or item.get("topic") or ""
            abstract_text = item.get("abstract") or ""
            authors_txt = _authors_block(item)
        else:
            title, abstract_text, authors_txt = "", str(item), ""
        results.append(extract_keywords_and_countries(title, abstract_text, authors_txt))
    return results

def results_keywords_only(results: List[Dict[str, List[str]]]) -> List[List[str]]:
    return [r["keywords"] for r in results]

# ===== Main =====
if __name__ == "__main__":
    filename = "aiche_sample.json"
    save_full_results_to = "full_results.json"
    save_keywords_only_to = None

    all_results = process_file(filename)

    print("FULL RESULTS:")
    print(json.dumps(all_results, ensure_ascii=False, indent=2))

    if save_full_results_to:
        with open(save_full_results_to, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

    kw_matrix = results_keywords_only(all_results)
    print("\nKEYWORDS ONLY (array of arrays):")
    print(json.dumps(kw_matrix, ensure_ascii=False, indent=2))

    if save_keywords_only_to:
        with open(save_keywords_only_to, "w", encoding="utf-8") as f:
            json.dump(kw_matrix, f, ensure_ascii=False, indent=2)


NameError: name '_apply_chat' is not defined

This is the recent open router code that i used

In [12]:
import os
import json
import re
import time
import logging
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

# =========================
# Logging Configuration
# =========================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("extraction_process.log"),
        logging.StreamHandler()  # FIXED: Changed from StreamHeader to StreamHandler
    ]
)
logger = logging.getLogger(__name__)

# =========================
# OpenRouter client setup
# =========================
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY4")
if not API_KEY:
    logger.error("Set OPENROUTER_API_KEY in your environment or .env")
    raise RuntimeError("Missing API Key")

client = OpenAI(
    api_key=API_KEY, 
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:3000",
        "X-Title": "AIChE Extractor"
    }
)

MODEL_ID = "meta-llama/llama-3.3-70b-instruct:free"

# =========================
# Prompts (UNCHANGED)
# =========================
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5 to 10 unique entries without explanation.
5. Each keyword should ideally be 1-3 words, but can be longer if it is a specific phrase but in rare cases.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

# =========================
# Helpers
# =========================
def _extract_json(text: str) -> Dict[str, Any]:
    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            block = m.group(0)
            return json.loads(block)
    except Exception as e:
        logger.warning(f"Failed to parse JSON from output: {e}")
    return {}

def _clean_list(xs) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if k and k.lower() not in seen:
            seen.add(k.lower())
            out.append(k)
    return out

def _authors_block(item: Dict[str, Any]) -> str:
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm = (a.get("name") or "").strip()
            af = (a.get("affiliation") or "").strip()
            if nm or af: parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)[:800]

def _call_openrouter(messages: List[Dict[str, str]], max_new_tokens: int = 512, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                temperature=0.0,
                max_tokens=max_new_tokens,
            )
            if hasattr(resp, 'choices') and len(resp.choices) > 0:
                return resp.choices[0].message.content or ""
        except Exception as e:
            logger.error(f"Attempt {attempt+1} failed: {e}")
            if "401" in str(e):
                logger.error("Authentication Error. Check API Key.")
            if attempt < retries - 1:
                time.sleep(5) 
    return ""

def extract_keywords_and_countries(title: str, abstract_text: str, authors_txt: str) -> Dict[str, List[str]]:
    user_content = User_Prompt_template.format(
        few_shot=FEW_SHOT.strip(),
        title_or_topic=(title or "").strip(),
        authors_block=(authors_txt or "").strip(),
        abstract_text=(abstract_text or "").strip()
    )
    messages = [
        {"role": "system", "content": System_Prompt.strip()},
        {"role": "user", "content": user_content.strip()},
    ]
    raw_output = _call_openrouter(messages)
    data = _extract_json(raw_output)
    return {
        "keywords": _clean_list(data.get("keywords", [])),
        "countries": _clean_list(data.get("countries", []))
    }

# =========================
# Folder Processing Logic
# =========================
def run_batch_extraction(input_folder: str, output_folder: str):
    if not os.path.exists(input_folder):
        logger.error(f"Input folder '{input_folder}' does not exist.")
        return

    os.makedirs(output_folder, exist_ok=True)
    json_files = sorted([f for f in os.listdir(input_folder) if f.endswith('.json')])
    
    if not json_files:
        logger.info(f"No JSON files found in {input_folder}")
        return

    logger.info(f"Starting batch process for {len(json_files)} files.")
    master_results = []

    for file_name in json_files:
        input_path = os.path.join(input_folder, file_name)
        logger.info(f"--- Processing File: {file_name} ---")
        
        try:
            with open(input_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            file_results = []
            for i, item in enumerate(data):
                title = item.get("title") or item.get("topic") or "No Title"
                abstract = item.get("abstract") or ""
                authors = _authors_block(item)
                
                logger.info(f"[{i+1}/{len(data)}] Extracting: {title[:40]}...")
                extracted = extract_keywords_and_countries(title, abstract, authors)
                
                file_results.append({
                    "title": title,
                    "authors": authors,
                    "keywords": extracted["keywords"],
                    "countries": extracted["countries"],
                    "source_file": file_name,
                    "Year": 2023 
                })
            
            # Save individual file result
            base_name = os.path.splitext(file_name)[0]
            individual_out = os.path.join(output_folder, f"{base_name}_extracted.json")
            with open(individual_out, "w", encoding="utf-8") as f:
                json.dump(file_results, f, indent=2)
            
            master_results.extend(file_results)
            logger.info(f"Finished {file_name}. Saved to {individual_out}")

        except Exception as e:
            logger.error(f"Failed to process {file_name}: {e}")

    # Final Combined Output
    master_out = os.path.join(output_folder, "final_combined_keywords.json")
    with open(master_out, "w", encoding="utf-8") as f:
        json.dump(master_results, f, indent=2)
    
    logger.info(f"BATCH COMPLETE. Final master file: {master_out}")

if __name__ == "__main__":
    # Update these paths to your actual folder names
    INPUT_DIR = "dummy" 
    OUTPUT_DIR = "openrouter/extracted_results"

    run_batch_extraction(INPUT_DIR, OUTPUT_DIR)

2026-02-06 01:08:34,855 - INFO - Starting batch process for 1 files.
2026-02-06 01:08:34,856 - INFO - --- Processing File: aiche_sample.json ---
2026-02-06 01:08:34,856 - INFO - [1/10] Extracting: 3a- Micro/Nanoengineered Adhesive Biomat...
2026-02-06 01:08:35,795 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-06 01:08:35,795 - INFO - Retrying request to /chat/completions in 0.396010 seconds
2026-02-06 01:08:36,486 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-06 01:08:36,486 - INFO - Retrying request to /chat/completions in 0.756919 seconds
2026-02-06 01:08:37,536 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-06 01:08:37,541 - ERROR - Attempt 1 failed: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/meta-llama/llama-3.3-70b-instruct/839b2e30-a1b4-4974-b980-3e

KeyboardInterrupt: 

this is the correct version before the above

In [ ]:
import os
import json
import re
import time
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

# =========================
# OpenRouter client setup
# =========================
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY1")
if not API_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY in your environment or .env")

# Initialize the client with corrected base_url and required headers
client = OpenAI(
    api_key=API_KEY, 
    base_url="https://openrouter.ai/api/v1",  # Corrected: Removed /chat/completions
    default_headers={
        "HTTP-Referer": "http://localhost:3000", # Required for OpenRouter verification
        "X-Title": "AIChE Extractor"            # Identification for your app
    }
)

# Model configuration
MODEL_ID = "mistralai/mistral-7b-instruct" 

# =========================
# Prompts (Using your exact templates)
# =========================
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5 to 10 unique entries without explanation.
5. Each keyword should ideally be 1-3 words, but can be longer if it is a specific phrase but in rare cases.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

# =========================
# Helpers
# =========================
def _extract_json(text: str) -> Dict[str, Any]:
    """Robustly extracts JSON from the LLM response."""
    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            block = m.group(0)
            return json.loads(block)
    except Exception:
        pass
    return {}

def _clean_list(xs) -> List[str]:
    """Cleans extracted list items and removes duplicates."""
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if k and k.lower() not in seen:
            seen.add(k.lower())
            out.append(k)
    return out

def _authors_block(item: Dict[str, Any]) -> str:
    """Formats author/affiliation data for the prompt."""
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm = (a.get("name") or "").strip()
            af = (a.get("affiliation") or "").strip()
            if nm or af: parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)[:800]

# =========================
# Robust OpenRouter Call
# =========================
def _call_openrouter(messages: List[Dict[str, str]], max_new_tokens: int = 512, retries: int = 3) -> str:
    """Calls OpenRouter with retry logic for rate limits and auth issues."""
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                temperature=0.0,
                max_tokens=max_new_tokens,
            )
            
            if hasattr(resp, 'choices') and len(resp.choices) > 0:
                return resp.choices[0].message.content or ""
            else:
                print(f"Warning: Unexpected response format on attempt {attempt+1}")
                
        except Exception as e:
            print(f"Error on attempt {attempt+1}: {e}")
            # Common 401 error check
            if "401" in str(e):
                print("Hint: Check your API key and credits. Ensure no extra spaces in .env")
            
            if attempt < retries - 1:
                time.sleep(5) 
    return ""

# =========================
# Core extraction
# =========================
def extract_keywords_and_countries(title: str, abstract_text: str, authors_txt: str) -> Dict[str, List[str]]:
    """Generates the prompt and calls the API."""
    user_content = User_Prompt_template.format(
        few_shot=FEW_SHOT.strip(),
        title_or_topic=(title or "").strip(),
        authors_block=(authors_txt or "").strip(),
        abstract_text=(abstract_text or "").strip()
    )

    messages = [
        {"role": "system", "content": System_Prompt.strip()},
        {"role": "user", "content": user_content.strip()},
    ]

    raw_output = _call_openrouter(messages)
    data = _extract_json(raw_output)

    return {
        "keywords": _clean_list(data.get("keywords", [])),
        "countries": _clean_list(data.get("countries", []))
    }

# =========================
# Batch processing
# =========================
def process_file(path: str) -> List[Dict[str, Any]]:
    """Processes entire JSON file of abstracts."""
    if not os.path.exists(path):
        print(f"Error: File {path} not found.")
        return []

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = []
    print(f"Processing {len(data)} items...")
    
    for i, item in enumerate(data):
        title = item.get("title") or item.get("topic") or "No Title"
        abstract = item.get("abstract") or ""
        authors = _authors_block(item)
        
        print(f"[{i+1}/{len(data)}] Extracting: {title[:50]}...")
        extracted = extract_keywords_and_countries(title, abstract, authors)
        
        results.append({
            "title": title,
            "authors": authors,
            "keywords": extracted["keywords"],
            "countries": extracted["countries"],
            "Year": 2023 
        })
        
    return results

# =========================
# Main Execution
# =========================
if __name__ == "__main__":
    INPUT_FILE = "aiche_sample.json" 
    OUTPUT_DIR = "openrouter/extracted_ll"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_results = process_file(INPUT_FILE)

    with open(os.path.join(OUTPUT_DIR, "full_results.json"), "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2)

    print(f"\nSuccess! Results saved to {OUTPUT_DIR}")

instead of taking a file as input take a folder as input and run on everyfile and after completing the file save the extracted keywords related to the file name in the oputput folder and after completing everthing save all the keywords in a final file 
have appropiate logging and donot change any prompt

In [ ]:
presenting_author = "Priyanka Bholanath Shukla | University of Pittsburgh | The University of Texas at El Paso | Central Michigan University"
universities = [u.strip() for u in presenting_author.split('|') if 'University' in u]

# 3. Geocode Universities
from geopy.geocoders import Nominatim
from time import sleep

geolocator = Nominatim(user_agent="keyword_map")
university_locations = {}

for uni in universities:
    location = geolocator.geocode(uni)
    if location:
        university_locations[uni] = (location.latitude, location.longitude)
    sleep(1)  # be kind to free geocoding APIs

# 4. Create DataFrame for Visualization
import pandas as pd

df = pd.DataFrame({
    "university": list(university_locations.keys()),
    "lat": [loc[0] for loc in university_locations.values()],
    "lon": [loc[1] for loc in university_locations.values()],
    # "keyword": keywords[:len(university_locations)],
    "author": ["Priyanka Bholanath Shukla"] * len(university_locations),
    "datetime": ["2023-11-06"] * len(university_locations)
})

# 5. Plotly World Map Visualization
import plotly.express as px

fig = px.scatter_geo(
    df,
    lat='lat',
    lon='lon',
    # text='keyword',
    hover_name='university',
    hover_data={ "author": True, "datetime": True},
    projection="natural earth"
)

fig.update_layout(title="Research Keyword Origins by University", title_x=0.5)
fig.show()


: 

In [1]:
import torch
import torchvision
from transformers import pipeline

print(torch.__version__)
print(torchvision.__version__)
print(torch.cuda.is_available())


c:\Users\chand.CHANDAN\anaconda3\envs\aiche-llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.5.1+cu121
0.20.1+cu121
True
